# Week 2 — Context Engineering I (Local with LiteLLM & Ollama)

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-02-context-engineering-i-content.html`.

**You will practice:**
1. System prompt vs. no system prompt, same question.
2. Zero-shot vs. few-shot prompting on a classification task.
3. Chain-of-thought vs. direct answer on a multi-step problem.
4. Schema-forced JSON output — LiteLLM, then a light ADK and LangChain preview.
5. Two open exercises.

**Environment:** Running 100% locally with Ollama, RTX GPU acceleration, and LiteLLM.

In [1]:
%pip install -q --upgrade litellm google-adk langchain-community langchain-ollama python-dotenv pydantic

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
from dotenv import load_dotenv
from litellm import completion
from pydantic import BaseModel

load_dotenv()
MODEL = "ollama_chat/qwen2.5:14b" 

## 1. System prompt vs. no system prompt

Same user question, with and without a system instruction.

In [6]:
question = "My bike's front brake feels loose, what should I check?"

no_system = completion(
    model=MODEL,
    messages=[{"role": "user", "content": question}],
)
print("--- Without system prompt ---")
print(no_system.choices[0].message.content)

with_system = completion(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a support assistant for a bike rental company. "
                "Only answer questions related to bike rentals, maintenance, and safety. "
                "Keep answers under 3 sentences. If asked something unrelated, politely decline."
            ),
        },
        {"role": "user", "content": question},
    ],
)
print("\n--- With system prompt ---")
print(with_system.choices[0].message.content)

--- Without system prompt ---
If your bike's front brake feels loose, there are a few key areas you should check to diagnose and potentially fix the issue:

1. **Brake Pads**: Check if the brake pads are worn down or improperly aligned. If the pads are worn down, they won't grip the rim effectively, and if they are misaligned, they might not be contacting the rim properly. Make sure the pads are correctly positioned and not worn out.

2. **Brake Cable Tension**: Ensure that the cable tension is adequate. A loose cable can cause a spongy feel and poor braking performance. Adjust the cable tension if it's too loose by tightening the cable at the brake lever or caliper.

3. **Brake Lever**: Check if the brake lever is loose or misaligned. If the lever is loose, it won't pull the cable effectively. If it's misaligned, it might not be pulling the cable at the right angle, affecting brake performance. Adjust the lever if necessary to improve its alignment and tighten it securely to the handl

## 2. Zero-shot vs. few-shot

Watch the output format stabilize once examples are added.

In [4]:
zero_shot = 'Classify the sentiment of this review as positive, negative, or neutral: "It\'s fine, does what it says on the box."'

few_shot = '''Classify the sentiment of a review as positive, negative, or neutral.

Review: "Fast shipping and the product works great."
Sentiment: positive

Review: "It arrived broken and support never replied."
Sentiment: negative

Review: "It's fine, does what it says on the box."
Sentiment:'''

res_zero = completion(model=MODEL, messages=[{"role": "user", "content": zero_shot}])
res_few = completion(model=MODEL, messages=[{"role": "user", "content": few_shot}])

print("Zero-shot:", res_zero.choices[0].message.content.strip())
print("Few-shot: ", res_few.choices[0].message.content.strip())

Zero-shot: The sentiment of the review "It's fine, does what it says on the box" can be classified as neutral. The phrase indicates that the product meets basic expectations without any notable positive or negative attributes.
Few-shot:  neutral


## 3. Chain-of-thought vs. direct answer

In [7]:
problem = "A store had 120 apples. It sold 35% of them in the morning and 28 more in the afternoon. How many apples are left?"

direct = completion(
    model=MODEL,
    messages=[{"role": "user", "content": problem + " Answer with just the number."}],
    temperature=0.1,
)
print("Direct:", direct.choices[0].message.content.strip())

cot = completion(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": problem + ' Think step by step, then give the final answer on its own line starting with "Answer:".',
        }
    ],
    temperature=0.1,
)
print("\nChain-of-thought:\n", cot.choices[0].message.content)

Direct: 44

Chain-of-thought:
 To determine how many apples are left, let's break down the problem step by step.

1. **Calculate the number of apples sold in the morning:**
   - The store sold 35% of 120 apples in the morning.
   - To find 35% of 120, we calculate \( 0.35 \times 120 = 42 \) apples.

2. **Calculate the number of apples remaining after the morning sales:**
   - Initially, there were 120 apples.
   - After selling 42 apples in the morning, the remaining apples are \( 120 - 42 = 78 \) apples.

3. **Calculate the number of apples remaining after the afternoon sales:**
   - In the afternoon, the store sold an additional 28 apples.
   - After selling 28 apples in the afternoon, the remaining apples are \( 78 - 28 = 50 \) apples.

Therefore, the number of apples left is 50.

Answer: 50


## 4. Schema-forced JSON output

### 4a. LiteLLM / Pydantic

In [11]:
class TicketTriage(BaseModel):
    category: str   # "billing" | "technical" | "account" | "other"
    urgency: str     # "low" | "medium" | "high"
    summary: str

response = completion(
    model=MODEL,
    messages=[
        {"role": "user", "content": "My card was charged twice for the same order and I need this fixed today."}
    ],
    response_format=TicketTriage,
)
ticket = TicketTriage.model_validate_json(response.choices[0].message.content)
print(ticket)
print(ticket.category, "|", ticket.urgency)

category='instruction' urgency='high' summary='Resolve double charging on credit card for a single order immediately.'
instruction | high


### 4b. Google ADK — `output_schema` (light preview with LiteLlm)

ADK's `Agent` accepts a Pydantic model directly via `output_schema` and returns validated structured output.

In [ ]:
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

triage_agent = Agent(
    model=LiteLlm(model=MODEL),
    name="triage_agent",
    instruction="Triage the support message into the given schema.",
    output_schema=TicketTriage,
)

async def ask_adk_agent(agent, prompt, app_name="week2_app", user_id="student"):
    session_service = InMemorySessionService()
    runner = Runner(
        agent=agent,
        app_name=app_name,
        session_service=session_service,
        auto_create_session=True,
    )
    session = await session_service.create_session(app_name=app_name, user_id=user_id)
    content = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    final_text = ""
    async for event in runner.run_async(user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text:
                    final_text += p.text
    return final_text

raw_json = await ask_adk_agent(triage_agent, "My card was charged twice for the same order and I need this fixed today.")
print(raw_json)
print(TicketTriage.model_validate_json(raw_json))

{
  "category": "Payment Issue",
  "urgency": "High",
  "summary": "Duplicate Charge for Order - Immediate Resolution Needed"
}
category='Payment Issue' urgency='High' summary='Duplicate Charge for Order - Immediate Resolution Needed'


### 4c. LangChain — `with_structured_output` (light preview)

In [12]:
from langchain_ollama import ChatOllama

ollama_model_name = MODEL.replace("ollama_chat/", "").replace("ollama/", "")
llm = ChatOllama(model=ollama_model_name)
structured_llm = llm.with_structured_output(TicketTriage)

result = structured_llm.invoke("My card was charged twice for the same order and I need this fixed today.")
print(result)

category='instruction' urgency='high' summary='Resolve duplicate charge on card for a single order immediately.'


## 5. Exercises

In [19]:
# TODO Exercise 1 — Design a few-shot prompt
# Pick a small labeling task of your own (e.g. classify emails as "spam"/"not spam",
# or tag a sentence with a difficulty level). Write a zero-shot version and a few-shot
# version (3-4 examples) and compare the outputs on 3 new inputs.

task_instructions = "Classify the following message as real, spam, or scam."

zero_shot_prompt = f"{task_instructions}\nMessage: {{message}}\nClassification:"

few_shot_prompt = f"""{task_instructions}

Message: "Hey, are we still meeting for lunch tomorrow at noon?"
Classification: Real

Message: "Congratulations! You've won a $1000 gift card, call 1-800-123-4567 to claim it now!!!"
Classification: Spam

Message: "Your bank account has been suspended. Send your ID and password to verify your identity immediately or lose access."
Classification: Scam

Message: {{message}}
Classification:"""

test_inputs = [
    "Reminder: your dentist appointment is on Friday at 3pm.",
    "You've been selected for a free iPhone! Text WIN to 12345 now to claim your prize.",
    "This is the IRS. You owe back taxes and must pay immediately using gift cards or you will be arrested.",
]

for message in test_inputs:
    zero_res = completion(
        model=MODEL,
        messages=[{"role": "user", "content": zero_shot_prompt.format(message=message)}],
        temperature=0.1,
    )
    few_res = completion(
        model=MODEL,
        messages=[{"role": "user", "content": few_shot_prompt.format(message=message)}],
        temperature=0.1,
    )
    print("================================================================================")
    print(f"Message: {message}")
    print(f"Zero-shot: {zero_res.choices[0].message.content.strip()}")
    print(f"Few-shot:  {few_res.choices[0].message.content.strip()}")

Message: Reminder: your dentist appointment is on Friday at 3pm.
Zero-shot: Based on the content of the message, it appears to be a reminder about a dentist appointment. Without additional context suggesting it is fraudulent or malicious, this message seems to be real. However, the classification can vary depending on how you received the message and whether you were expecting it. If you were expecting a reminder from your dentist and the contact details are legitimate, it is likely real. If you did not schedule an appointment or the message seems to come from an unusual source, it could be spam or a scam attempting to gather personal information or redirect you to a fraudulent website. Given the limited information, I would classify this as real, but caution is advised if the message seems out of place.
Few-shot:  Classification: Real
Message: You've been selected for a free iPhone! Text WIN to 12345 now to claim your prize.
Zero-shot: Classification: Scam

This message is likely a sc

In [23]:
# TODO Exercise 2 — Mini "resume line" parser
# Define a Pydantic model `ResumeLine` with fields like `role: str`, `company: str`,
# `years: float`. Use schema-forced JSON output to parse a single free-text line such as:
#   "Senior backend engineer at Northwind Traders for 3.5 years"
# into a ResumeLine instance, and print the parsed object.

class ResumeLine(BaseModel):
    name: str
    restaurant: str
    cuisine_specialty: str
    years_experience: float
    michelin_stars: int

chef_text = (
    "Chef Massimo Bottura leads Osteria Francescana in Modena, specializing in modern "
    "Italian cuisine, with over 30 years of experience and 3 Michelin stars."
)

response = completion(
    model=MODEL,
    messages=[
        {"role": "user", "content": f"Parse the following chef description into structured data:\n{chef_text}"}
    ],
    response_format=ResumeLine,
)

parsed_chef = ResumeLine.model_validate_json(response.choices[0].message.content)
print("Parsed ResumeLine instance:")
print(parsed_chef)
print(
    f"Name: {parsed_chef.name} | Restaurant: {parsed_chef.restaurant} | "
    f"Specialty: {parsed_chef.cuisine_specialty} | Experience: {parsed_chef.years_experience} years | "
    f"Michelin Stars: {parsed_chef.michelin_stars}"
)

Parsed ResumeLine instance:
name='Massimo Bottura' restaurant='Osteria Francescana' cuisine_specialty='Modern Italian' years_experience=30.0 michelin_stars=3
Name: Massimo Bottura | Restaurant: Osteria Francescana | Specialty: Modern Italian | Experience: 30.0 years | Michelin Stars: 3


## Next week

Week 3 — **Context Engineering II**: managing the context window (summarization, compression, sliding window)
and an introduction to embeddings and semantic search. See `week-03-context-engineering-ii-content.html`.